# VARC (Vision ARC) — Offline Training + Test-Time Training on Kaggle

This notebook runs the **official, unmodified** code from [lillian039/VARC](https://github.com/lillian039/VARC),
the reference implementation for the paper *["ARC Is a Vision Problem!"](https://arxiv.org/abs/2511.14761)*
(Hu et al.). It:

1. Clones the repo as-is (no code edits — same `ARC_ViT.py`, `ARC_loader.py`, `offline_train_ARC.py`,
   `test_time_train_ARC.py` you already have).
2. Runs **offline training** of the ViT model on the ARC-AGI-1 training set (+ optional RE-ARC data).
3. Builds the augmented per-task **test-time-training (TTT)** dataset and runs TTT + inference on a
   configurable set of ARC-AGI-1 evaluation tasks.
4. Aggregates predictions into Pass@1 / Pass@2 / Oracle scores, the same metrics the repo's own
   `analysis.py` reports.

### About scale
The authors trained on **8×H200 GPUs** (offline: ~5h; TTT: all 400 eval tasks run in parallel across
8 GPUs). Kaggle gives you a **single GPU** (T4/P100, ~9h/session, ~30h/week quota). The code is left
completely untouched — only the **command-line hyperparameters** in the Config cell below are yours to
tune. Everything is exposed there: epochs, batch size, RE-ARC usage, number of TTT tasks, etc.
Defaults are set to the values from the repo's own `script/*.sh` files; lower `OFFLINE_EPOCHS`,
`REARC_LIMIT`, and `NUM_TTT_TASKS` for a quick smoke test.


## 0. Check GPU
Make sure the Kaggle notebook's **Accelerator** is set to a GPU (Settings → Accelerator → GPU T4 x2 / P100).

In [1]:
!nvidia-smi
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Wed Jul 15 18:04:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
print(os.listdir('.'))

['__notebook__.ipynb']


## 1. Clone the official VARC repo (unmodified)

The ARC-AGI-1 and RE-ARC raw data ship inside the repo itself, so no separate data download is needed.

In [3]:
import os, subprocess

WORK_DIR = "/kaggle/working"
REPO_DIR = f"{WORK_DIR}/VARC"

os.chdir(WORK_DIR)
if not os.path.isdir(REPO_DIR):
    result = subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/lillian039/VARC.git"],
        cwd=WORK_DIR, capture_output=True, text=True,
    )
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0 or not os.path.isdir(REPO_DIR):
        raise RuntimeError(
            "git clone failed, so the repo was not created. The #1 cause on Kaggle is that "
            "'Internet' is turned off for this notebook. Fix: Notebook Settings (top right, "
            "three-dot menu or side panel) -> Internet -> toggle ON, then re-run this cell. "
            "You may also need to verify your Kaggle account (phone verification) to enable internet access."
        )
else:
    print("Repo already present, skipping clone.")

os.chdir(REPO_DIR)
!ls



Cloning into 'VARC'...
Updating files:  85% (2042/2379)
Updating files:  86% (2046/2379)
Updating files:  87% (2070/2379)
Updating files:  88% (2094/2379)
Updating files:  89% (2118/2379)
Updating files:  90% (2142/2379)
Updating files:  91% (2165/2379)
Updating files:  92% (2189/2379)
Updating files:  93% (2213/2379)
Updating files:  93% (2216/2379)
Updating files:  94% (2237/2379)
Updating files:  95% (2261/2379)
Updating files:  96% (2284/2379)
Updating files:  97% (2308/2379)
Updating files:  98% (2332/2379)
Updating files:  99% (2356/2379)
Updating files: 100% (2379/2379)
Updating files: 100% (2379/2379), done.

analysis.py	 offline_train_ARC.py  requirements.txt  test_time_train_ARC.py
assets		 raw_data	       script		 utils
augment_data.py  README.md	       src


## 2. Install dependencies

Kaggle already ships a CUDA-enabled PyTorch + NumPy, so we deliberately **don't** force the exact
`torch==2.7.0` / `numpy==2.2.6` pins from `requirements.txt` (doing so can break the pre-configured GPU
driver stack). Everything else from `requirements.txt` is installed as-is.

In [4]:
!pip install -q timm==1.0.12 einops huggingface_hub "wandb==0.22.0" diffusers datasets tqdm
print("Done.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.6/51.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.6/19.6 MB 71.1 MB/s eta 0:00:00
Done.


## 3. Configuration (edit this cell)

Every hyperparameter used by `offline_train_ARC.py` and `test_time_train_ARC.py` is defined here.
Defaults for the **model / offline-training** block mirror `script/offline_train_VARC_ViT.sh`
(the exact config used to train VARC-ViT-18M in the paper). Defaults for **TTT** mirror
`script/test_time_training_VARC_ViT_ARC1.sh`. Change anything you like — the launch cells below just
forward these variables as CLI flags to the untouched scripts.

In [5]:
import sys, os, subprocess, json, time, glob

os.chdir(REPO_DIR)

# ----------------------------------------------------------------------------------
# Model architecture (must be identical for offline training and TTT)
# ----------------------------------------------------------------------------------
ARCHITECTURE   = "vit"     # "vit" or "unet"
IMAGE_SIZE     = 64        # canvas size
PATCH_SIZE     = 2         # ViT patch size
EMBED_DIM      = 512
DEPTH          = 10
NUM_HEADS      = 8
NUM_COLORS     = 12        # 10 ARC colors + background-canvas color + border/shape token

# torch.compile
NO_COMPILE = True

# ----------------------------------------------------------------------------------
# Dataset Controls (ARC-AGI-1, ARC-AGI-2, and NVARC)
# ----------------------------------------------------------------------------------
# We will combine these datasets into one unified folder
COMBINED_DATA_ROOT = "raw_data/combined_dataset"
TRAIN_SPLIT        = "training"

# Toggle which datasets to train on
INCLUDE_ARC1       = True   # Train on standard ARC-AGI-1 training dataset
INCLUDE_ARC2       = True   # Train on official ARC-AGI-2 training dataset
INCLUDE_NVARC      = True   # Train on NVARC dataset

# Precise control panel for NVARC
NVARC_TASK_LIMIT   = 65     # How many unique base tasks to select (-1 for ALL)
NVARC_AUG_LIMIT    = 5      # How many augmentations to load per task (0 for base only)

# ----------------------------------------------------------------------------------
# Offline training parameters
# ----------------------------------------------------------------------------------
OFFLINE_EPOCHS      = 30          
OFFLINE_BATCH_SIZE  = 8
LEARNING_RATE        = 3e-4
WEIGHT_DECAY          = 0
LR_SCHEDULER          = "cosine"
INCLUDE_REARC         = True       
REARC_LIMIT           = 5         
NUM_WORKERS            = 2
VIS_EVERY              = 50
SAVE_PATH              = "saves/offline_train_ViT/checkpoint_final.pt"
BEST_SAVE_PATH         = "saves/offline_train_ViT/checkpoint_best.pt"

USE_WANDB              = False     
WANDB_PROJECT          = "VisionARC"
WANDB_RUN_NAME         = "offline_train_VARC_kaggle"

# ----------------------------------------------------------------------------------
# Test-time training (TTT)
# ----------------------------------------------------------------------------------
TTT_EPOCHS         = 100
TTT_BATCH_SIZE     = 8
TTT_NUM_ATTEMPTS   = 10     
TTT_NUM_EACH       = 1      
TTT_EVAL_SAVE_NAME = "ARC_1_eval_ViT"   
NUM_TTT_TASKS      = 3

ALL_ARC1_EVAL_TASKS = [
    "af24b4cc", "e1d2900e", "903d1b4a", "4e469f39", "b1fc8b8e", "2c737e39", "992798f6", "00576224", "48131b3c", "60a26a3e", 
    "59341089", "31d5ba1a", "e633a9e5", "62ab2642", "73c3b0d8", "c663677b", "c48954c1", "08573cc6", "136b0064", "929ab4e9", 
    "5b526a93", "ef26cbf6", "fafd9572", "67c52801", "ad7e01d0", "506d28a5", "27a77e38", "d492a647", "72a961c9", "fd4b2b02"
]

TTT_TASKS = ALL_ARC1_EVAL_TASKS[:NUM_TTT_TASKS]
print(f"Will build combined dataset, run offline training for {OFFLINE_EPOCHS} epochs, then TTT on {len(TTT_TASKS)} task(s).")

Will build combined dataset, run offline training for 30 epochs, then TTT on 3 task(s).


## 4. Helper: run a command and stream its output live

In [6]:
import sys

def run_streaming(cmd, cwd=REPO_DIR, env=None):
    """Run `cmd` (a list of args) and stream stdout/stderr to the notebook live,
    including the scripts' \r-updating epoch progress bars (train_loss/train_acc/
    eval_acc/eta etc. get printed once per epoch; the % bar updates continuously
    within an epoch)."""
    print("Running:", " ".join(cmd))
    full_env = os.environ.copy()
    full_env["PYTHONUNBUFFERED"] = "1"   # make the child flush immediately, not just on newline
    if env:
        full_env.update(env)

    proc = subprocess.Popen(
        cmd, cwd=cwd, env=full_env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    # Read char-by-char (not line-by-line) so \r progress-bar updates render live
    # instead of being buffered until the next real \n.
    while True:
        ch = proc.stdout.read(1)
        if ch == "" and proc.poll() is not None:
            break
        if ch:
            sys.stdout.write(ch)
            sys.stdout.flush()

    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {' '.join(cmd)}")
    return proc.returncode


In [7]:
# ==========================================
# NEW CELL: Prepare the Combined Dataset
# ==========================================
import os
import shutil
import glob

# Set up clean target layout for the training files
target_train_dir = f"{COMBINED_DATA_ROOT}/data/training"
if os.path.exists(target_train_dir):
    shutil.rmtree(target_train_dir)
os.makedirs(target_train_dir, exist_ok=True)

# ----------------------------------------------------------------------------------
# DIRECT KAGGLE RE-ARC MAPPING & COUNTING
# ----------------------------------------------------------------------------------
KAGGLE_REARC_PATH = "/kaggle/input/datasets/jeffreyjian/re-arc/re_arc/tasks" 

target_rearc = f"{COMBINED_DATA_ROOT}/re_arc"
if os.path.exists(target_rearc) or os.path.islink(target_rearc):
    try:
        os.remove(target_rearc)
    except IsADirectoryError:
        shutil.rmtree(target_rearc)

rearc_count = 0
if os.path.exists(KAGGLE_REARC_PATH):
    os.makedirs(COMBINED_DATA_ROOT, exist_ok=True)
    os.symlink(os.path.abspath(KAGGLE_REARC_PATH), os.path.abspath(target_rearc))
    
    # Count the number of JSON task files in the RE-ARC directory
    rearc_files = glob.glob(os.path.join(KAGGLE_REARC_PATH, "*.json"))
    rearc_count = len(rearc_files)
    print(f"✅ Successfully linked Kaggle RE-ARC input ({KAGGLE_REARC_PATH}) to training pipeline.")
    print(f"-> Found {rearc_count} RE-ARC augmentation tasks available.")
else:
    print(f"❌ [Error] RE-ARC not found at {KAGGLE_REARC_PATH}. Double-check your Kaggle input name!")

# ----------------------------------------------------------------------------------
# Merge ARC-AGI-1, ARC-AGI-2, and NVARC
# ----------------------------------------------------------------------------------
arc1_count = 0
print("\nStep 1: Merging ARC-AGI-1 training tasks...")
if INCLUDE_ARC1:
    arc1_files = glob.glob("raw_data/ARC-AGI/data/training/*.json")
    for fp in arc1_files:
        shutil.copy(fp, target_train_dir)
    arc1_count = len(arc1_files)
    print(f"-> Copied {arc1_count} ARC-AGI-1 training tasks.")

arc2_count = 0
print("\nStep 2: Merging ARC-AGI-2 training tasks...")
if INCLUDE_ARC2:
    arc2_paths = [
        "raw_data/ARC-AGI-2/data/training", 
        "/kaggle/input/arc-agi-2/data/training",
        "/kaggle/input/datasets/boristown/arc-agi-2/training",
        "/kaggle/input/arc-prize-2025/arc-agi-2/data/training"
    ]
    arc2_dir = next((path for path in arc2_paths if os.path.exists(path)), None)
    if arc2_dir:
        arc2_files = glob.glob(os.path.join(arc2_dir, "*.json"))
        for fp in arc2_files:
            shutil.copy(fp, target_train_dir)
        arc2_count = len(arc2_files)
        print(f"-> Copied {arc2_count} ARC-AGI-2 training tasks.")
    else:
        print("-> [Info] ARC-AGI-2 directory not found. Skipping.")

nvarc_base_copied = 0
nvarc_augs_copied = 0
print("\nStep 3: Merging NVARC tasks with fine controls...")
if INCLUDE_NVARC:
    try:
        task_limit = int(NVARC_TASK_LIMIT)
        aug_limit = int(NVARC_AUG_LIMIT)
    except (NameError, ValueError):
        task_limit = 200
        aug_limit = 15

    print(f"-> Active Task Limit: {task_limit}")
    print(f"-> Active Augmentation Limit: {aug_limit}")
    
    nvarc_dir = None
    possible_paths = [
        "/kaggle/input/nvarc-synthetic-puzzles/nvarc_full",
        "/kaggle/input/sorokin-nvarc-synthetic-puzzles/nvarc_full",
        "/kaggle/input/datasets/sorokin/nvarc-synthetic-puzzles/nvarc_full",
        "/kaggle/input/nvarc-synthetic-puzzles",
        "raw_data/nvarc_full"
    ]
    
    for path in possible_paths:
        if os.path.exists(path):
            try:
                if len(os.listdir(path)) > 0:
                    nvarc_dir = path
                    break
            except Exception:
                continue

    if nvarc_dir and os.path.exists(nvarc_dir):
        print(f"-> Auto-detected active NVARC path at: {nvarc_dir}")
        
        try:
            raw_contents = os.listdir(nvarc_dir)
            print(f"-> Total raw items found in directory: {len(raw_contents)}")
        except Exception as e:
            print(f"❌ Failed to read directory contents: {e}")
            raw_contents = []

        all_folders = [
            f for f in raw_contents 
            if not f.startswith('.') 
            and not f.endswith('.json') 
            and not f.endswith('.txt') 
            and not f.endswith('.csv')
            and not f.endswith('.md')
        ]
        all_folders.sort()
        print(f"-> Filtered down to {len(all_folders)} prospective task folders")
        
        if 'target_train_dir' not in locals() and 'target_train_dir' not in globals():
            target_train_dir = "raw_data/combined_dataset/data/training"
        os.makedirs(target_train_dir, exist_ok=True)
        
        if task_limit > 0:
            target_folders = all_folders[:task_limit]
        elif task_limit == 0:
            target_folders = []
            print("⚠️ WARNING: NVARC_TASK_LIMIT is set to 0. No folders will be processed.")
        else:
            target_folders = all_folders
            
        tasks_processed = 0
        nvarc_copied = 0
        max_files_per_task = 1 + aug_limit
        
        print(f"-> Sliced to {len(target_folders)} target folders. Starting processing...")

        for folder_name in target_folders:
            folder_path = os.path.join(nvarc_dir, folder_name)
            
            try:
                all_files = [f for f in os.listdir(folder_path) if f.endswith(".json")]
                all_files.sort()
            except Exception as e:
                print(f"❌ Error listing files in folder {folder_name}: {e}")
                continue
            
            files_to_copy = all_files[:max_files_per_task]
            copied_from_this_task = 0
            
            for file_name in files_to_copy:
                file_path = os.path.join(folder_path, file_name)
                
                try:
                    with open(file_path, 'r') as f:
                        task_data = json.load(f)
                    
                    formatted_task = None
                    
                    # Scenario A: Already a valid standard ARC dictionary
                    if isinstance(task_data, dict) and 'train' in task_data:
                        formatted_task = task_data
                    
                    # Scenario B: It's a raw list of grid-pairs/dictionaries
                    elif isinstance(task_data, list) and len(task_data) > 0:
                        normalized_pairs = []
                        for item in task_data:
                            if isinstance(item, dict) and "input" in item and "output" in item:
                                normalized_pairs.append(item)
                            elif isinstance(item, (list, tuple)) and len(item) == 2:
                                normalized_pairs.append({"input": item[0], "output": item[1]})
                        
                        if len(normalized_pairs) > 0:
                            # Standardize format: split last item into 'test', rest to 'train'
                            formatted_task = {
                                "train": normalized_pairs[:-1] if len(normalized_pairs) > 1 else normalized_pairs,
                                "test": [normalized_pairs[-1]]
                            }
                    
                    if formatted_task:
                        # Write standardized dictionary to target directory
                        out_path = os.path.join(target_train_dir, f"nvarc_{file_name}")
                        with open(out_path, 'w') as out_f:
                            json.dump(formatted_task, out_f)
                        
                        nvarc_copied += 1
                        copied_from_this_task += 1
                    else:
                        print(f"⚠️ Skipped {file_name}: JSON structure invalid/empty (type={type(task_data)})")
                except Exception as e:
                    print(f"❌ Error copying/formatting {file_name}: {e}")
                    continue
            
            if copied_from_this_task > 0:
                tasks_processed += 1
                    
        print(f"-> Successfully loaded and standardized NVARC.")
        print(f"   - Selected Task Folders: {tasks_processed}")
        print(f"   - Total Verified NVARC JSONs Standardized: {nvarc_copied}")
    else:
        print(f"❌ [Error] Could not find any non-empty 'nvarc_full' directory inside /kaggle/input/.")
# ==================================================================================
# ACCURATE SAMPLES (GRID PAIRS) COUNT & SUMMARY
# ==================================================================================
print("\nAnalyzing dataset to calculate precise training grid pairs...")

def count_grids_in_dir(directory, prefix_filter=None):
    """Counts the total input/output training grid pairs in a directory of JSON files."""
    total_grids = 0
    file_list = glob.glob(os.path.join(directory, "*.json"))
    for fp in file_list:
        filename = os.path.basename(fp)
        # Skip if a prefix filter is provided and does not match
        if prefix_filter and not filename.startswith(prefix_filter):
            continue
        try:
            with open(fp, 'r') as f:
                task_data = json.load(f)
                # Count pairs in the 'train' list of the task
                if isinstance(task_data, dict) and 'train' in task_data:
                    total_grids += len(task_data['train'])
                elif isinstance(task_data, list): # fallback for alternative formats
                    total_grids += len(task_data)
        except Exception:
            pass
    return total_grids

# 1. ARC-AGI-1 Grid count
arc1_grids = 0
if INCLUDE_ARC1:
    # Read directly from copied files that do not belong to NVARC
    arc1_grids = count_grids_in_dir(target_train_dir)
    # Exclude NVARC grids if they are in the same directory
    if INCLUDE_NVARC:
        nvarc_grids_in_target = count_grids_in_dir(target_train_dir, prefix_filter="nvarc_")
        arc1_grids -= nvarc_grids_in_target

# 2. ARC-AGI-2 Grid count
arc2_grids = 0
if INCLUDE_ARC2 and arc2_dir:
    arc2_grids = count_grids_in_dir(arc2_dir)

# 3. NVARC Grid count
nvarc_grids = 0
if INCLUDE_NVARC:
    # Read the count directly from the copied files in the target directory
    nvarc_grids = count_grids_in_dir(target_train_dir, prefix_filter="nvarc_")

# 4. RE-ARC Grid count (Based on unique base tasks and your limit per task)
rearc_grids = 0
if os.path.exists(target_rearc):
    rearc_task_count = len(os.listdir(target_rearc))
    # Each RE-ARC task will have REARC_LIMIT variations used during training
    rearc_grids = rearc_task_count * REARC_LIMIT

total_combined_grids = arc1_grids + arc2_grids + nvarc_grids + rearc_grids

# Output the precise breakdown
print("=" * 64)
print(f"{'PRECISE TRAINING GRID PAIRS SUMMARY':^64}")
print("=" * 64)
print(f"  • ARC-AGI-1 Training Grids  : {arc1_grids:<6} pairs")
print(f"  • ARC-AGI-2 Training Grids  : {arc2_grids:<6} pairs")
print(f"  • NVARC Training Grids      : {nvarc_grids:<6} pairs (Base + Augs)")
print(f"  • RE-ARC Training Grids     : {rearc_grids:<6} pairs ({rearc_task_count} tasks × {REARC_LIMIT} limit)")
print("-" * 64)
print(f"  Total Combined Training Grids: {total_combined_grids} input/output pairs")
print("=" * 64)

✅ Successfully linked Kaggle RE-ARC input (/kaggle/input/datasets/jeffreyjian/re-arc/re_arc/tasks) to training pipeline.
-> Found 400 RE-ARC augmentation tasks available.

Step 1: Merging ARC-AGI-1 training tasks...
-> Copied 400 ARC-AGI-1 training tasks.

Step 2: Merging ARC-AGI-2 training tasks...
-> Copied 1000 ARC-AGI-2 training tasks.

Step 3: Merging NVARC tasks with fine controls...
-> Active Task Limit: 65
-> Active Augmentation Limit: 5
-> Auto-detected active NVARC path at: /kaggle/input/datasets/sorokin/nvarc-synthetic-puzzles/nvarc_full
-> Total raw items found in directory: 120
-> Filtered down to 120 prospective task folders
-> Sliced to 65 target folders. Starting processing...
-> Successfully loaded and standardized NVARC.
   - Selected Task Folders: 65
   - Total Verified NVARC JSONs Standardized: 390

Analyzing dataset to calculate precise training grid pairs...
              PRECISE TRAINING GRID PAIRS SUMMARY               
  • ARC-AGI-1 Training Grids  : 3260   pai

## 5. Offline training

Calls `offline_train_ARC.py` exactly as `script/offline_train_VARC_ViT.sh` does, minus `torchrun`
(single GPU here instead of 8). This trains the ViT from scratch on ARC-AGI-1 training tasks
(+ RE-ARC if enabled) and evaluates each epoch on held-out ARC-AGI-1 training-task test pairs,
saving the best checkpoint to `BEST_SAVE_PATH`.

In [8]:
# ==========================================
# UPDATED CELL: Run Offline Training
# ==========================================
os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

offline_cmd = [
    sys.executable, "offline_train_ARC.py",
    "--epochs", str(OFFLINE_EPOCHS),
    "--depth", str(DEPTH),
    "--batch-size", str(OFFLINE_BATCH_SIZE),
    "--image-size", str(IMAGE_SIZE),
    "--patch-size", str(PATCH_SIZE),
    "--learning-rate", str(LEARNING_RATE),
    "--weight-decay", str(WEIGHT_DECAY),
    "--embed-dim", str(EMBED_DIM),
    "--num-heads", str(NUM_HEADS),
    "--num-colors", str(NUM_COLORS),
    "--data-root", COMBINED_DATA_ROOT, # Directs parser to our generated data mix!
    "--train-split", TRAIN_SPLIT,
    "--save-path", SAVE_PATH,
    "--best-save-path", BEST_SAVE_PATH,
    "--lr-scheduler", LR_SCHEDULER,
    "--architecture", ARCHITECTURE,
    "--vis-every", str(VIS_EVERY),
    "--num-workers", str(NUM_WORKERS),
]

if INCLUDE_REARC:
    offline_cmd += ["--include-rearc", "--rearc-limit", str(REARC_LIMIT)]
if USE_WANDB:
    offline_cmd += ["--use-wandb", "--wandb-project", WANDB_PROJECT, "--wandb-run-name", WANDB_RUN_NAME]
if NO_COMPILE:
    offline_cmd += ["--no-compile"]

# Reduce CUDA memory fragmentation
mem_env = {"PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True", "PYTORCH_ALLOC_CONF": "expandable_segments:True"}

t0 = time.time()
run_streaming(offline_cmd, env=mem_env)
print(f"\nOffline training finished in {(time.time()-t0)/60:.1f} min")

Running: /usr/bin/python3 offline_train_ARC.py --epochs 30 --depth 10 --batch-size 8 --image-size 64 --patch-size 2 --learning-rate 0.0003 --weight-decay 0 --embed-dim 512 --num-heads 8 --num-colors 12 --data-root raw_data/combined_dataset --train-split training --save-path saves/offline_train_ViT/checkpoint_final.pt --best-save-path saves/offline_train_ViT/checkpoint_best.pt --lr-scheduler cosine --architecture vit --vis-every 50 --num-workers 2 --include-rearc --rearc-limit 5 --no-compile
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached t

In [9]:
assert os.path.exists(BEST_SAVE_PATH), "Best checkpoint not found — check the training logs above."
size_mb = os.path.getsize(BEST_SAVE_PATH) / 1e6
print(f"Best checkpoint saved at {BEST_SAVE_PATH} ({size_mb:.1f} MB)")


Best checkpoint saved at saves/offline_train_ViT/checkpoint_best.pt (222.6 MB)


## 6. Build the augmented per-task TTT dataset

Reproduces `augment_data.py` (called exactly the same way, just imported instead of run as `__main__`):
for every ARC-AGI-1 evaluation task it writes an `eval_color_permute_ttt_9/<task_id>/` folder containing
the task's demonstration pairs plus 9 color-permuted copies, which `test_time_train_ARC.py` treats as
the per-task training set for TTT.

In [10]:
import os
import json
import shutil
import time
from utils.data_augmentation import augment_raw_data_split_per_task

# --- Configuration & Paths ---
ARC_PRIZE_2024_DIR = "/kaggle/input/competitions/arc-prize-2024"
ARC_AGI_2_DIR = "/kaggle/input/datasets/boristown/arc-agi-2"
DATA_ROOT = "raw_data/ARC-AGI"

TTT_TASK_SOURCES = {
    # ARC-1 task from ARC Prize 2024
    "e1d2900e": {
        "type": "bulk",
        "challenges_file": f"{ARC_PRIZE_2024_DIR}/arc-agi_evaluation_challenges.json",
        "solutions_file": f"{ARC_PRIZE_2024_DIR}/arc-agi_evaluation_solutions.json"
    },
    # ARC-2 tasks directly from your ARC-AGI-2 folder list
    "e3721c99": {
        "type": "individual",
        "file_path": f"{ARC_AGI_2_DIR}/evaluation/e3721c99.json"
    },    
    "d35bdbdc": {
        "type": "individual",
        "file_path": f"{ARC_AGI_2_DIR}/evaluation/d35bdbdc.json"
    }
}

TTT_TASKS = list(TTT_TASK_SOURCES.keys())

# --- Stage Tasks ---
t0 = time.time()
staging_dir = f"{DATA_ROOT}/data/evaluation"
os.makedirs(staging_dir, exist_ok=True)

print("-> Extracting and preparing task files for TTT...")
for task_id, source in TTT_TASK_SOURCES.items():
    target_path = f"{staging_dir}/{task_id}.json"
    
    if source["type"] == "bulk":
        ch_path = source["challenges_file"]
        sol_path = source["solutions_file"]
        
        if not os.path.exists(ch_path) or not os.path.exists(sol_path):
            raise FileNotFoundError(f"Missing bulk files for {task_id} at {ch_path} or {sol_path}")
            
        print(f"   Extracting {task_id} from bulk dataset...")
        with open(ch_path, "r") as f:
            challenges = json.load(f)
        with open(sol_path, "r") as f:
            solutions = json.load(f)
            
        if task_id in challenges:
            task_data = challenges[task_id]
            for idx, test_item in enumerate(task_data["test"]):
                test_item["output"] = solutions[task_id][idx]
                
            with open(target_path, "w") as f:
                json.dump(task_data, f, indent=4)
            print(f"   Successfully extracted and staged bulk task: {task_id}")
        else:
            raise KeyError(f"Task {task_id} not found in the bulk challenges file.")
            
    elif source["type"] == "individual":
        src_path = source["file_path"]
        if not os.path.exists(src_path):
            raise FileNotFoundError(f"Individual file not found for {task_id} at {src_path}")
            
        shutil.copy2(src_path, target_path)
        print(f"   Successfully copied individual task: {task_id}")

# --- Generate Color Permutations ---
print("\n-> Running augmentation on staged tasks...")
augment_raw_data_split_per_task(
    dataset_root=DATA_ROOT,
    split="evaluation",
    output_subdir="eval_color_permute_ttt_9",
    num_permuate=9,
    only_basic=True,
)

print(f"\nAugmented TTT dataset built in {(time.time()-t0)/60:.1f} min")

# Verification
for task_name in TTT_TASKS:
    task_dir = f"{DATA_ROOT}/data/eval_color_permute_ttt_9/{task_name}"
    if os.path.exists(task_dir):
        print(f"✓ Task {task_name}: Found {len(os.listdir(task_dir))} augmented files.")
    else:
        print(f"⚠️ Warning: Task folder {task_dir} is missing.")

-> Extracting and preparing task files for TTT...
   Extracting e1d2900e from bulk dataset...
   Successfully extracted and staged bulk task: e1d2900e
   Successfully copied individual task: e3721c99
   Successfully copied individual task: d35bdbdc

-> Running augmentation on staged tasks...
5 augmenters will be applied
[augment] Completed split 'evaluation': 20100 augmented tasks prepared in raw_data/ARC-AGI/data/eval_color_permute_ttt_9/ff72ca3e

Augmented TTT dataset built in 0.6 min
✓ Task e1d2900e: Found 51 augmented files.
✓ Task e3721c99: Found 51 augmented files.
✓ Task d35bdbdc: Found 51 augmented files.


## 7. Test-time training (TTT)

For each task in `TTT_TASKS`, calls `test_time_train_ARC.py` exactly as
`script/test_time_training_VARC_ViT_ARC1.sh` does per task (minus the outer GPU-parallel loop, since
we have one GPU). Each call: fine-tunes a fresh copy of the offline-trained model on that task's
augmented demonstrations (`TTT_NUM_EACH` independent times), then predicts the held-out test pair(s)
with `TTT_NUM_ATTEMPTS` augmented views per attempt and majority-votes the answer. Predictions and a
per-task Pass@1/Pass@2/Oracle report (via the repo's own `analyze_prediction.py`) are printed live and
saved under `outputs/{TTT_EVAL_SAVE_NAME}_attempt_{0..TTT_NUM_EACH-1}/`.

In [11]:
import sys
import time

def build_ttt_cmd(task_name):
    return [
        sys.executable, "test_time_train_ARC.py",
        "--epochs", str(TTT_EPOCHS),
        "--depth", str(DEPTH),
        "--batch-size", str(TTT_BATCH_SIZE),
        "--image-size", str(IMAGE_SIZE),
        "--patch-size", str(PATCH_SIZE),
        "--learning-rate", str(LEARNING_RATE),
        "--weight-decay", str(WEIGHT_DECAY),
        "--embed-dim", str(EMBED_DIM),
        "--num-heads", str(NUM_HEADS),
        "--num-colors", str(NUM_COLORS),
        "--resume-checkpoint", BEST_SAVE_PATH,
        "--resume-skip-task-token",
        "--lr-scheduler", LR_SCHEDULER,
        "--train-split", f"eval_color_permute_ttt_9/{task_name}",
        "--eval-split", f"eval_color_permute_ttt_9/{task_name}",
        "--data-root", DATA_ROOT,
        "--architecture", ARCHITECTURE,
        "--eval-save-name", TTT_EVAL_SAVE_NAME,
        "--num-attempts", str(TTT_NUM_ATTEMPTS),
        "--ttt-num-each", str(TTT_NUM_EACH),
    ] + (["--no-compile"] if NO_COMPILE else [])

t0 = time.time()
for i, task_name in enumerate(TTT_TASKS, 1):
    print(f"\n===== [{i}/{len(TTT_TASKS)}] TTT on task {task_name} =====")
    run_streaming(build_ttt_cmd(task_name), env=mem_env)
print(f"\nTTT finished for {len(TTT_TASKS)} task(s) in {(time.time()-t0)/60:.1f} min")


===== [1/3] TTT on task e1d2900e =====
Running: /usr/bin/python3 test_time_train_ARC.py --epochs 100 --depth 10 --batch-size 8 --image-size 64 --patch-size 2 --learning-rate 0.0003 --weight-decay 0 --embed-dim 512 --num-heads 8 --num-colors 12 --resume-checkpoint saves/offline_train_ViT/checkpoint_best.pt --resume-skip-task-token --lr-scheduler cosine --train-split eval_color_permute_ttt_9/e1d2900e --eval-split eval_color_permute_ttt_9/e1d2900e --data-root raw_data/ARC-AGI --architecture vit --eval-save-name ARC_1_eval_ViT --num-attempts 10 --ttt-num-each 1 --no-compile
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alia

In [12]:
import os
import json
from utils.eval_utils import get_majority_vote

# --- Configuration ---
save_name = TTT_EVAL_SAVE_NAME
attempt_idx = 0  # Which TTT run execution to inspect (0 to TTT_NUM_EACH-1)

COLOR_MAP = {
    0: "\033[48;5;0m  \033[0m",    # Black
    1: "\033[48;5;21m  \033[0m",   # Blue
    2: "\033[48;5;196m  \033[0m",  # Red
    3: "\033[48;5;46m  \033[0m",   # Green
    4: "\033[48;5;226m  \033[0m",  # Yellow
    5: "\033[48;5;244m  \033[0m",  # Gray
    6: "\033[48;5;201m  \033[0m",  # Magenta
    7: "\033[48;5;208m  \033[0m",  # Orange
    8: "\033[48;5;51m  \033[0m",   # Cyan
    9: "\033[48;5;88m  \033[0m",   # Maroon
    10: "\033[48;5;239m  \033[0m", # Background canvas token
    11: "\033[48;5;255m  \033[0m", # Border / extra token
}

def render_grid_row(grid, row_idx):
    """Renders a single row of a grid as colored blocks. Returns spacer blocks if row_idx is out of bounds."""
    if not grid or row_idx >= len(grid):
        width = len(grid[0]) if (grid and len(grid) > 0) else 3
        return "  " * width
    return "".join(COLOR_MAP.get(cell, "??") for cell in grid[row_idx])

def print_grids_horizontal(input_grid, gt_grid, pred_grid):
    """Prints the Input, Ground Truth, and Predicted grids side-by-side."""
    h_in = len(input_grid) if input_grid else 0
    h_gt = len(gt_grid) if gt_grid else 0
    h_pred = len(pred_grid) if pred_grid else 0
    max_h = max(h_in, h_gt, h_pred)
    
    # Grid headers
    w_in = len(input_grid[0]) if h_in > 0 else 5
    w_gt = len(gt_grid[0]) if h_gt > 0 else 5
    w_pred = len(pred_grid[0]) if h_pred > 0 else 5
    
    header_in = "INPUT".center(w_in * 2)
    header_gt = "GROUND TRUTH".center(w_gt * 2)
    header_pred = "PREDICTION".center(w_pred * 2)
    
    print(f"  {header_in}      {header_gt}      {header_pred}")
    print(f"  {'-'*(w_in*2)}      {'-'*(w_gt*2)}      {'-'*(w_pred*2)}")
    
    # Draw side-by-side row by row
    for r in range(max_h):
        str_in = render_grid_row(input_grid, r)
        str_gt = render_grid_row(gt_grid, r)
        str_pred = render_grid_row(pred_grid, r)
        print(f"  {str_in}    |    {str_gt}    |    {str_pred}")

# --- Run visualization over all tasks ---
print(f"==================================================================")
print(f"=== VISUALIZING TEST TIME TRAINING (TTT) OUTCOMES FOR ALL TASKS ===")
print(f"==================================================================\n")

for task_name in TTT_TASKS:
    pred_path = f"outputs/{save_name}_attempt_{attempt_idx}/{task_name}_predictions.json"
    gt_path = f"{DATA_ROOT}/data/evaluation/{task_name}.json"
    
    print(f"\n##################################################################")
    print(f" TASK: {task_name}")
    print(f"##################################################################")
    
    if not os.path.exists(pred_path):
        print(f"⚠️ Predictions missing at {pred_path}. Skipping visualization for this task.")
        continue
        
    try:
        with open(pred_path) as f:
            predictions_dict = json.load(f)
        with open(gt_path) as f:
            gt_data = json.load(f)
    except Exception as e:
        print(f"❌ Error loading data files for {task_name}: {e}")
        continue
        
    for idx_str, preds in sorted(predictions_dict.items(), key=lambda x: int(x[0])):
        test_idx = int(idx_str)
        input_grid = gt_data["test"][test_idx]["input"]
        gt_grid = gt_data["test"][test_idx]["output"]
        
        # Resolve majority voted prediction
        mv = get_majority_vote(preds)
        predicted_grid = mv[0]["prediction"] if mv else None
        
        print(f"\n--- [Task {task_name}] Test Pair Index: {idx_str} ---")
        print_grids_horizontal(input_grid, gt_grid, predicted_grid)
        
        status = "SUCCESS ✅" if predicted_grid == gt_grid else "FAILED ❌"
        print(f"Result Status: {status}\n")

=== VISUALIZING TEST TIME TRAINING (TTT) OUTCOMES FOR ALL TASKS ===


##################################################################
 TASK: e1d2900e
##################################################################

--- [Task e1d2900e] Test Pair Index: 0 ---
                             INPUT                                                          GROUND TRUTH                                                       PREDICTION                         
  ------------------------------------------------------------      ------------------------------------------------------------      ------------------------------------------------------------
                                                                  |                                                                    |                                                                
                                                                  |                                                                    |         

## 8. Aggregate results

Merges predictions across the `TTT_NUM_EACH` independent TTT runs per task and computes overall
Pass@1 / Pass@2 / Oracle, the same scoring logic as the repo's `utils/analyze_prediction.py` /
`analysis.py` (majority vote via `get_majority_vote`), generalized to whatever `TTT_TASKS` you ran.

In [13]:
from utils.eval_utils import get_majority_vote

def collect_results(task_list, save_name, num_each, data_root):
    task_type = data_root.split("/")[-1]  # e.g. "ARC-AGI"
    all_task_num = correct_1 = correct_2 = correct_oracle = 0
    per_task_scores = {}

    for task_name in task_list:
        merged = None
        for i in range(num_each):
            fp = f"outputs/{save_name}_attempt_{i}/{task_name}_predictions.json"
            if not os.path.exists(fp):
                continue
            with open(fp) as f:
                d = json.load(f)
            if merged is None:
                merged = {k: list(v) for k, v in d.items()}
            else:
                for k, v in d.items():
                    merged.setdefault(k, [])
                    merged[k] += v

        if merged is None:
            print(f"[warn] no predictions found for {task_name}, skipping")
            continue

        gt_path = f"raw_data/{task_type}/data/evaluation/{task_name}.json"
        with open(gt_path) as f:
            gt_data = json.load(f)
        ground_truth = {str(i): item["output"] for i, item in enumerate(gt_data["test"])}

        n_examples = len(merged)
        t1 = t2 = torac = 0
        for idx_str, preds in merged.items():
            mv = get_majority_vote(preds)
            gt = ground_truth[str(idx_str)]
            p1 = bool(mv) and mv[0]["prediction"] == gt
            p2 = p1 or (len(mv) > 1 and mv[1]["prediction"] == gt)
            oracle = any(e["prediction"] == gt for e in mv)
            t1 += p1; t2 += p2; torac += oracle

        task_pass1 = t1 / max(n_examples, 1)
        task_pass2 = t2 / max(n_examples, 1)
        task_oracle = torac / max(n_examples, 1)
        per_task_scores[task_name] = dict(pass_at_1=task_pass1, pass_at_2=task_pass2, oracle=task_oracle)

        all_task_num += 1
        correct_1 += task_pass1
        correct_2 += task_pass2
        correct_oracle += task_oracle

        status = "\u2705" if (task_pass1 or task_pass2) else "\u274c"
        print(f"{task_name}: pass@1={task_pass1:.2f}  pass@2={task_pass2:.2f}  oracle={task_oracle:.2f}  {status}")

    if all_task_num:
        print(f"\n==== Aggregate over {all_task_num} task(s) ====")
        print(f"Pass@1: {correct_1/all_task_num:.4f}")
        print(f"Pass@2: {correct_2/all_task_num:.4f}")
        print(f"Oracle: {correct_oracle/all_task_num:.4f}")
    return per_task_scores

scores = collect_results(TTT_TASKS, TTT_EVAL_SAVE_NAME, TTT_NUM_EACH, DATA_ROOT)


e1d2900e: pass@1=0.00  pass@2=0.00  oracle=0.00  ❌
e3721c99: pass@1=0.00  pass@2=0.00  oracle=0.00  ❌
d35bdbdc: pass@1=0.00  pass@2=0.00  oracle=0.00  ❌

==== Aggregate over 3 task(s) ====
Pass@1: 0.0000
Pass@2: 0.0000
Oracle: 0.0000


## 9. Scaling up to the full paper setting

To move closer to the numbers reported in the paper / README (Pass@1 ≈ 52-56 on ARC-AGI-1 with
VARC-ViT-18M), in the **Config cell**:

- Set `REARC_LIMIT = -1` and keep `OFFLINE_EPOCHS = 100` (this alone will take many single-GPU hours —
  the paper's 5h12m figure is on 8×H200).
- Set `NUM_TTT_TASKS = 400` (or assign `TTT_TASKS = ALL_ARC1_EVAL_TASKS` directly) to run TTT on every
  ARC-AGI-1 evaluation task, matching `script/test_time_training_VARC_ViT_ARC1.sh`. Expect this to take
  a long time run sequentially on one GPU — consider splitting `ALL_ARC1_EVAL_TASKS` into chunks across
  multiple Kaggle sessions (checkpointing is already done via `BEST_SAVE_PATH`, so you can re-run just
  the TTT section any time after offline training finishes) or, if you have 2 GPUs (T4 x2), running two
  notebook processes each covering half of `ALL_ARC1_EVAL_TASKS` with `CUDA_VISIBLE_DEVICES` set per
  process (see `script/test_time_training_VARC_ViT_ARC1.sh` for the pattern).
- For ARC-AGI-2 instead, swap `DATA_ROOT` to `"raw_data/ARC-AGI-2"` and re-run the augmentation +
  TTT cells; the offline-trained ARC-AGI-1 checkpoint is reused as-is per the README.

Citation:
```
@misc{hu2025arcvisionproblem,
      title={{ARC} Is a Vision Problem!},
      author={Keya Hu and Ali Cy and Linlu Qiu and Xiaoman Delores Ding and Runqian Wang and Yeyin Eva Zhu and Jacob Andreas and Kaiming He},
      year={2025},
      eprint={2511.14761},
      archivePrefix={arXiv},
      primaryClass={cs.CV},
      url={https://arxiv.org/abs/2511.14761},
}
```
